# API Wikipedia Retriever + Dynamic Query Rewriting
Instead of directly searching a complex trivia question, the class uses a provided LLM (specifically a Qwen model) to rewrite the question into an optimized Wikipedia search query. 
It supports several strategies:
* **entity**: Extracts the main person, place, or concept.
* **mock_answer**: Frames the question as an incomplete sentence to find direct factual lookups.
* **keywords**: Extracts 3-8 essential keywords for wordy questions.
* **auto**: Prompts the LLM to dynamically classify the question and pick the best strategy above.

1. The retriever queries the Wikipedia API for related article titles
2. The titles are used to get the complete article bodies
3. Article bodies are chunked and then both hybrid retrieval and reranking are used
4. The best 3 chunks are included in the prompt for context

In [ ]:
import requests
import re
import torch
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder


class WikipediaRetriever:
    """Wikipedia retriever with optional Qwen-based query rewriting and
    choice-aware cross-encoder reranking for multiple-choice questions.

    Plug-and-play usage:

        retriever = WikipediaRetriever(
            qwen_model=model,
            qwen_tokenizer=tokenizer,
            num_results=3,
            max_chars_per_doc=500
        )

        results = retriever.retrieve(
            query=question,
            choices=answers,       # optional list of strings
            strategy="entity",     # default | keywords | mock_answer | entity
        )
    """

    SEARCH_URL = "https://en.wikipedia.org/w/api.php"
    HEADERS = {
        "User-Agent": "PoliMillionaire-NLP-Project/1.0 (educational coursework)"
    }

    def __init__(
        self,
        num_results=5,
        max_chars_per_doc=1500,
        timeout=5.0,
        biencoder_model="sentence-transformers/all-MiniLM-L6-v2",
        crossencoder_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        bm25_candidates=30,
        qwen_model=None,
        qwen_tokenizer=None,
        llm_max_input_chars=1200,
        llm_max_new_tokens=24,
    ):
        self.num_results = num_results
        self.max_chars = max_chars_per_doc
        self.timeout = timeout
        self.bm25_candidates = bm25_candidates

        self.qwen_model = qwen_model
        self.qwen_tokenizer = qwen_tokenizer
        self.llm_max_input_chars = llm_max_input_chars
        self.llm_max_new_tokens = llm_max_new_tokens

        self.session = requests.Session()
        self.session.headers.update(self.HEADERS)

        self._search_cache = {}
        self._fullpage_cache = {}
        self._llm_query_cache = {}

        self.biencoder = SentenceTransformer(biencoder_model)
        self.crossencoder = CrossEncoder(crossencoder_model)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=100,
            length_function=len,
        )

    # ------------------------------------------------------------------ #
    # Qwen utilities                                                     #
    # ------------------------------------------------------------------ #
    def _has_qwen(self):
        return self.qwen_model is not None and self.qwen_tokenizer is not None

    def _format_choices(self, choices):
        return "\n".join(f"{chr(65+i)}) {c}" for i, c in enumerate(choices))

    def _sanitize_search_query(self, text, max_words=10):
        text = text.strip()
        text = re.sub(r"^[A-D][\)\.\:\-]\s*", "", text)
        text = re.sub(r"^Answer\s*:\s*", "", text, flags=re.I)
        text = re.sub(r"^Search query\s*:\s*", "", text, flags=re.I)
        text = re.sub(r"^Entity\s*:\s*", "", text, flags=re.I)
        text = re.sub(r"[^A-Za-z0-9À-ÿ'\"\-–—\(\) ]+", " ", text)
        text = " ".join(text.split())
        return " ".join(text.split()[:max_words])

    def _generate_with_qwen(self, system_prompt, user_prompt):
        if not self._has_qwen():
            raise ValueError("Qwen model/tokenizer not provided.")

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        text = self.qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_device = next(self.qwen_model.parameters()).device
        inputs = self.qwen_tokenizer(
            [text],
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        )
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.qwen_model.generate(
                **inputs,
                max_new_tokens=self.llm_max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=self.qwen_tokenizer.eos_token_id,
            )

        generated_ids = outputs[:, inputs["input_ids"].shape[1]:]
        response = self.qwen_tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        return response

    def _build_search_query(self, question, choices=None, strategy="default"):
        cache_key = (question, tuple(choices) if choices else None, strategy)
        if cache_key in self._llm_query_cache:
            return self._llm_query_cache[cache_key]

        # Fallback / default: don't use Qwen
        if strategy == "default" or not choices or not self._has_qwen():
            result = question
            self._llm_query_cache[cache_key] = result
            return result

        question = question[:self.llm_max_input_chars]
        choices_text = self._format_choices(choices)[:self.llm_max_input_chars]

    # ------------------------------------------------------------------ #
        # 1. Dynamic Strategy Routing (The "Auto" Agent)
        # ------------------------------------------------------------------ #
        if strategy == "auto":
            auto_system_prompt = (
                "You are an expert search query routing agent. Classify the trivia question into one of three query strategies:\n"
                "1. 'entity' - Best when the question is focused on a specific person, place, historical event, or named concept.\n"
                "2. 'mock_answer' - Best when the question is a direct factual lookup that can easily be framed as an incomplete sentence.\n"
                "3. 'keywords' - Best when the question is complex, wordy, or requires combining multiple distinct concepts.\n\n"
                "Reply with ONLY one exact word: entity, mock_answer, or keywords."
            )
            auto_user_prompt = f"Question:\n{question}\n\nStrategy:"
            
            try:
                # Ask the LLM to pick the strategy
                chosen_strategy = self._generate_with_qwen(auto_system_prompt, auto_user_prompt).strip().lower()
                
                # Sanitize output just in case the LLM adds punctuation
                chosen_strategy = re.sub(r'[^a-z_]', '', chosen_strategy)
                
                if chosen_strategy not in ["keywords", "mock_answer", "entity"]:
                    chosen_strategy = "entity"  # Safe default if hallucination occurs
                    
            except Exception as e:
                print(f"[Auto-Router] Routing failed, falling back to 'entity': {e}")
                chosen_strategy = "entity"
                
            print(f"[Auto-Router] Selected query strategy: {chosen_strategy}")
            strategy = chosen_strategy # Override the strategy variable for the next block

        # ------------------------------------------------------------------ #
        # 2. Query Generation (Using the chosen or provided strategy)
        # ------------------------------------------------------------------ #
        system_prompt = (
            "You rewrite quiz questions into short Wikipedia search queries. "
            "Return only the requested text and nothing else."
        )

        if strategy == "keywords":
            user_prompt = (
                f"Question:\n{question}\n\n"
                f"Choices:\n{choices_text}\n\n"
                "Extract the best short Wikipedia search query using 3 to 8 keywords. "
                "Return only the query."
            )
        elif strategy == "mock_answer":
            user_prompt = (
                f"Question:\n{question}\n\n"
                "Given the question, give a mock answer containing everything but the response. "
                "Examples:\n"
                "- Which is the capital of Australia? -> The capital of Australia is\n"
                "- What is 3 + 2? -> 3 + 2 is\n"
                "- How old is Barack Obama? -> Barack Obama is x years old\n\n"
                "Return only the mock answer text."
            )
        elif strategy == "entity":
            user_prompt = (
                f"Question:\n{question}\n\n"
                "Extract the single most important entity, topic, person, place, work, "
                "or concept needed to answer this question. "
                "Return only that entity or topic."
            )
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        try:
            # Generate the actual search query using the determined strategy
            raw = self._generate_with_qwen(system_prompt, user_prompt)
            result = self._sanitize_search_query(raw)
            if not result:
                result = question
        except Exception as e:
            print(f"Qwen query generation failed, falling back to full question. Error: {e}")
            result = question

        self._llm_query_cache[cache_key] = result
        return result

    # ------------------------------------------------------------------ #
    # Wikipedia search/fetch                                             #
    # ------------------------------------------------------------------ #
    def _search(self, query):
        if query in self._search_cache:
            return self._search_cache[query]

        params = {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
            "srlimit": self.num_results,
            "srprop": "",
        }

        try:
            r = self.session.get(self.SEARCH_URL, params=params, timeout=self.timeout)
            r.raise_for_status()
            hits = r.json().get("query", {}).get("search", [])
            titles = [h["title"] for h in hits]
        except Exception:
            titles = []

        self._search_cache[query] = titles
        return titles

    def _fetch_full_pages(self, titles_list):
        if not titles_list:
            return {}

        uncached_titles = [t for t in titles_list if t not in self._fullpage_cache]
        if uncached_titles:
            titles_string = "|".join(uncached_titles)
            params = {
                "action": "query",
                "titles": titles_string,
                "prop": "extracts",
                "explaintext": True,
                "exsectionformat": "plain",
                "format": "json",
                "utf8": 1,
            }
            try:
                r = self.session.get(self.SEARCH_URL, params=params, timeout=self.timeout)
                r.raise_for_status()
                pages = r.json().get("query", {}).get("pages", {})
                for page_data in pages.values():
                    title = page_data.get("title")
                    extract = page_data.get("extract", "")
                    if title is not None:
                        self._fullpage_cache[title] = extract
            except Exception as e:
                print(f"Wikipedia API Error: {e}")

        return {title: self._fullpage_cache.get(title, "") for title in titles_list}

    # ------------------------------------------------------------------ #
    # Chunking + retrieval                                               #
    # ------------------------------------------------------------------ #
    def _chunk(self, text, title):
        raw_chunks = self.splitter.split_text(text)
        return [{"title": title, "chunk": c} for c in raw_chunks]

    def _hybrid_retrieve(self, query, all_chunks):
        texts = [c["chunk"] for c in all_chunks]
        if not texts:
            return []

        k = min(self.bm25_candidates, len(texts))

        tokenized = [t.lower().split() for t in texts]
        bm25 = BM25Okapi(tokenized)
        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_top_idx = set(np.argsort(bm25_scores)[-k:][::-1])

        query_emb = self.biencoder.encode(query, convert_to_numpy=True)
        chunk_embs = self.biencoder.encode(texts, convert_to_numpy=True)
        cosine_scores = np.dot(chunk_embs, query_emb) / (
            np.linalg.norm(chunk_embs, axis=1) * np.linalg.norm(query_emb) + 1e-9
        )
        bienc_top_idx = set(np.argsort(cosine_scores)[-k:][::-1])

        candidate_idx = list(bm25_top_idx | bienc_top_idx)
        return [all_chunks[i] for i in candidate_idx]

    # ------------------------------------------------------------------ #
    # Choice-aware cross-encoder reranking                               #
    # ------------------------------------------------------------------ #
    def _rerank(self, query, candidates, choices=None):
        """
        If choices are provided, use a multiple-choice style scoring:

          - For each candidate chunk and each choice, build:
              "Question: {query}\nChoice: {choice}\nPassage: {chunk}"
          - Score with cross-encoder.
          - For each candidate chunk, take the max score over choices.
          - Rank candidates by that score.

        If choices is None or empty, fall back to scoring (query, chunk).
        """
        if not candidates:
            return []

        # Fallback: original behavior (no choices)
        if not choices:
            pairs = [(query, c["chunk"]) for c in candidates]
            scores = self.crossencoder.predict(pairs)
            ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
            return [c for _, c in ranked]

        # Multiple-choice: let the cross-encoder see the choice as well
        # Build all (query+choice, chunk) pairs
        pair_texts = []
        pair_meta = []  # (cand_idx, choice_idx)

        for ci, cand in enumerate(candidates):
            chunk_text = cand["chunk"]
            for ai, choice in enumerate(choices):
                qc_text = (
                    f"Question: {query}\n"
                    f"Choice: {choice}\n"
                    f"Passage: {chunk_text}"
                )
                pair_texts.append((qc_text, ""))  # CrossEncoder expects (text1, text2)
                pair_meta.append((ci, ai))

        # Score all pairs
        scores = self.crossencoder.predict(pair_texts)

        # For each candidate chunk, keep the best score over choices
        cand_best_scores = np.full(len(candidates), -1e9, dtype=np.float32)
        for (cand_idx, choice_idx), score in zip(pair_meta, scores):
            if score > cand_best_scores[cand_idx]:
                cand_best_scores[cand_idx] = score

        # Sort candidates by their best score
        ranked_indices = np.argsort(cand_best_scores)[::-1]
        return [candidates[i] for i in ranked_indices]

    # ------------------------------------------------------------------ #
    # Public API                                                         #
    # ------------------------------------------------------------------ #
    def retrieve(self, query, choices=None, strategy="default"):
        search_query = self._build_search_query(
            question=query,
            choices=choices,
            strategy=strategy,
        )

        print(f"SEARCH QUERY [{strategy}]: {search_query}")

        titles = self._search(search_query)
        print(f"TITLES: {titles}")

        all_chunks = []
        pages_dict = self._fetch_full_pages(titles[: self.num_results])

        for title, full_text in pages_dict.items():
            if full_text:
                all_chunks.extend(self._chunk(full_text, title))

        if not all_chunks:
            return []

        # Keep retrieval grounded in the ORIGINAL question
        candidates = self._hybrid_retrieve(query, all_chunks)
        ranked = self._rerank(query, candidates, choices=choices)

        out = []
        for item in ranked:
            out.append({
                "title": item["title"],
                "summary": item["chunk"][: self.max_chars],
            })
            if len(out) == self.num_results:
                break

        return out

    def format_context(self, retrievals):
        if not retrievals:
            return ""
        parts = [f"[{r['title']}] {r['summary']}" for r in retrievals]
        return "\n\n".join(parts)

    def retrieve_and_format(self, query, choices=None, strategy="auto"):
        retrievals = self.retrieve(query=query, choices=choices, strategy=strategy)
        return self.format_context(retrievals)

    def cache_stats(self):
        return {
            "searches_cached": len(self._search_cache),
            "fullpages_cached": len(self._fullpage_cache),
            "llm_queries_cached": len(self._llm_query_cache),
        }


# Example module-level instance, assuming `model` and `tokenizer` are defined
retriever = WikipediaRetriever(
    num_results=3,
    max_chars_per_doc=500,
    timeout=5.0,
    qwen_model=model,
    qwen_tokenizer=tokenizer,
)

# Quick smoke test (no choices, still works)
_test_ctx = retriever.retrieve_and_format("capital of Australia")
print("retriever sanity test:")
print(f"  context length: {len(_test_ctx)} chars")
print(f"  preview: {_test_ctx[:200]!r}")
print(f"  cache: {retriever.cache_stats()}")

# Two-Stage Workflow

This function orchestrates the multiple-choice answering process using a fallback mechanism:

**Stage 1: Parametric Memory (Zero Retrieval)**

    The function first attempts to answer the question using ask_llm with RAG disabled (use_rag=False).

    It relies purely on the LLM's pre-trained (parametric) memory.

    It checks the model's confidence score for its top answer. If the confidence meets or exceeds a strict threshold (default 0.99), the function immediately returns the answer, bypassing retrieval entirely to save time and compute.

**Stage 2: RAG Fallback**

    If the model's confidence in Stage 1 is below the threshold, the code prints an adaptive trigger message and falls back to Stage 2.

    It calls ask_llm again, but this time with RAG enabled (use_rag=True), allowing the model to ground its final answer using the Wikipedia retrieval tools.

In [68]:
# ==========================================
# LangChain Imports & Agentic Definitions
# ==========================================
from typing import Any, List, Optional
from langchain_core.language_models.llms import LLM
from langchain_core.callbacks.manager import CallbackManagerForLLMRun 
from langchain_core.prompts import PromptTemplate # <-- Updated import path
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor, create_react_agent 
from pydantic import PrivateAttr
import time

# 1. Minimal LangChain LLM Adapter for the existing HF model
class LocalHFLLM(LLM):
    """
    Wraps the global HuggingFace model and generation pipeline into a LangChain-compatible LLM.
    This avoids loading the model twice into memory.
    """
    _model: Any = PrivateAttr()
    _gen_config: dict = PrivateAttr()

    def __init__(self, model, gen_config, **kwargs):
        super().__init__(**kwargs)
        self._model = model
        self._gen_config = gen_config

    @property
    def _llm_type(self) -> str:
        return "local_hf"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        # Utilize the existing notebook infrastructure for inference
        raw_output, _, _ = _generate_with_messages(
            self._model,
            system_prompt="You are a helpful and precise logical agent.",
            user_prompt=prompt,
            gen_config=self._gen_config
        )
        return raw_output

# 2. RAG Retriever Tool for LangChain
@tool
def rag_retriever_tool_stage2(query: str) -> str:
    """
    Searches Wikipedia for the given query and returns relevant summaries.
    Useful for looking up factual information to resolve multiple-choice options.
    """
    # Relies on the globally instantiated `retriever`
    return retriever.retrieve_and_format(query, strategy="default")

# 3. Helper to isolate the top 2 options
def _top2_options(conf_dict: dict) -> List[int]:
    """Returns the 1-based indices of the two most probable options."""
    sorted_opts = sorted(conf_dict.items(), key=lambda x: x[1], reverse=True)
    return [sorted_opts[0][0], sorted_opts[1][0]]

# 4. The Main Agentic Entry Point
def ask_llm_agentic(model, question, option_texts, competition_id=None,
                    strategy=None, query_strategy="auto",
                    confidence_threshold=0.99):
    """
    Simplified Two-Stage Adaptive Pipeline:
      Stage 1: Direct prediction relying on parametric memory (use_rag=False).
      Stage 2: If max confidence < confidence_threshold, fallback to standard RAG (use_rag=True).
    """
    if strategy is None:
        strategy = P0_BASELINE

    # ==========================================
    # STAGE 1: Baseline Prediction (Zero Retrieval Overhead)
    # ==========================================
    stage1_result = ask_llm(model, question, option_texts, 
                            competition_id=competition_id, 
                            strategy=strategy, 
                            use_rag=False)
    
    best_ans = max(stage1_result["confidence"], key=stage1_result["confidence"].get)
    best_conf = stage1_result["confidence"][best_ans]

    if best_conf >= confidence_threshold:
        # High confidence -> Return immediately
        stage1_result["stage"] = 1
        return stage1_result

    print(f"\n[Adaptive] Stage 1 Confidence ({best_conf:.2f}) + ({best_ans}) < Threshold ({confidence_threshold}). Entering Stage 2 (RAG)...")

    # ==========================================
    # STAGE 2: Standard RAG Fallback
    # ==========================================
    stage2_result = ask_llm(model, question, option_texts,
                            competition_id=competition_id,
                            strategy=strategy,
                            use_rag=True, 
                            query_strategy=query_strategy)
    
    # Combine latency so the logs accurately reflect the total time spent across both stages
    stage2_result["latency"] += stage1_result["latency"]
    stage2_result["stage"] = 2
    
    return stage2_result

# Example run

In [54]:
question = "What was the primary punishment that slaves feared in the Republican era of ancient Rome?"
options = ["Execution of crucifixion", "Forced military service", "Flogging and imprisonment", "Hard labor in chains at quarries or mills"]
ask_llm_agentic(model, question, options, query_strategy="auto")

Question:
What was the primary punishment that slaves feared in the Republican era of ancient Rome?

Options:
1. Execution of crucifixion
2. Forced military service
3. Flogging and imprisonment
4. Hard labor in chains at quarries or mills

Answer:

[Adaptive] Stage 1 Confidence (0.92) < Threshold (0.99). Entering Stage 2 (RAG)...
[Auto-Router] Selected query strategy: entity
SEARCH QUERY [auto]: Roman slavery punishments
TITLES: ['Slavery in ancient Rome', 'Slavery', 'Slavery in antiquity']
Use the following reference information from Wikipedia to help you answer the question. If the references are not relevant, rely on your own knowledge.

References:
[Slavery] The treatment of slaves in the United States varied widely depending on conditions, times, and places. The power relationships of slavery corrupted many whites who had authority over slaves, with children showing their own cruelty. Masters and overseers resorted to physical punishments to impose their wills. Slaves were punishe

{'answer': 3,
 'confidence': {1: 0.06792271769714281,
  2: 4.88644061552422e-10,
  3: 0.932077273178523,
  4: 8.63569024187017e-09},
 'raw_output': '3',
 'latency': 2.9442780017852783,
 'rag_latency': 3.0938973426818848,
 'samples': None,
 'stage': 2}